# Molecular Electrolyte Analysis

Cross-scale analysis: solvent pairs, salt-solvent combos, and full electrolyte
formulations scored against cathodes/anodes.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import json

csv_path = '../showcase/outputs/electrolyte_compatibility.csv'
json_path = '../showcase/outputs/electrolyte_electrode_cross.json'

if not os.path.exists(csv_path):
    print('Generating data...')
    from showcase.electrolyte_analysis import main
    main()

df = pd.read_csv(csv_path)
with open(json_path) as f:
    cross_data = json.load(f)

solvent_pairs = df[df['pair_type'] == 'solvent-solvent']
salt_solvents = df[df['pair_type'] == 'salt-solvent']

print(f'Solvent-solvent pairs: {len(solvent_pairs)}')
print(f'Salt-solvent pairs: {len(salt_solvents)}')
print(f'Formulation-electrode combos: {len(cross_data.get("results", []))}')

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Solvent-Solvent Compatibility Heatmap
solvents = sorted(set(solvent_pairs['component_a']) | set(solvent_pairs['component_b']))
matrix = pd.DataFrame(np.nan, index=solvents, columns=solvents)

for _, row in solvent_pairs.iterrows():
    matrix.loc[row['component_a'], row['component_b']] = row['total_score']
    matrix.loc[row['component_b'], row['component_a']] = row['total_score']

np.fill_diagonal(matrix.values, 1.0)

fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(matrix.values, cmap='RdYlGn', vmin=0.5, vmax=1.0)
ax.set_xticks(range(len(solvents)))
ax.set_xticklabels(solvents, rotation=45, ha='right')
ax.set_yticks(range(len(solvents)))
ax.set_yticklabels(solvents)
ax.set_title('Solvent-Solvent Compatibility (Molecular Bridge)')
plt.colorbar(im, label='Score')

for i in range(len(solvents)):
    for j in range(len(solvents)):
        val = matrix.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7)

plt.tight_layout()
plt.show()

In [ ]:
# Salt-Solvent Scoring Matrix
salts = sorted(salt_solvents['component_a'].unique())
solvents_ss = sorted(salt_solvents['component_b'].unique())

ss_matrix = salt_solvents.pivot(index='component_a', columns='component_b',
                                 values='total_score')

fig, ax = plt.subplots(figsize=(10, 4))
im = ax.imshow(ss_matrix.values, cmap='RdYlGn', vmin=0.5, vmax=1.0, aspect='auto')
ax.set_xticks(range(len(ss_matrix.columns)))
ax.set_xticklabels(ss_matrix.columns, rotation=45, ha='right')
ax.set_yticks(range(len(ss_matrix.index)))
ax.set_yticklabels(ss_matrix.index)
ax.set_title('Salt-Solvent Compatibility')
plt.colorbar(im, label='Score')

for i in range(len(ss_matrix.index)):
    for j in range(len(ss_matrix.columns)):
        val = ss_matrix.values[i, j]
        if not np.isnan(val):
            ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Cross-scale: best formulation per electrode
print('Best Electrolyte Formulation per Electrode (Cross-Bridge)')
print('=' * 60)
for electrode, info in cross_data.get('best_per_electrode', {}).items():
    status = 'VIABLE' if info['viable'] else 'FAIL'
    print(f"  {electrode:12s} -> {info['formulation']} "
          f"(score={info['score']:.4f}) [{status}]")

In [ ]:
# Formulation-Electrode heatmap
results = cross_data.get('results', [])
if results:
    cross_df = pd.DataFrame(results)
    scored = cross_df.dropna(subset=['overall_score'])
    if not scored.empty:
        pivot = scored.pivot(index='formulation', columns='electrode',
                             values='overall_score')

        fig, ax = plt.subplots(figsize=(12, 5))
        im = ax.imshow(pivot.values, cmap='RdYlGn', vmin=0.4, vmax=1.0, aspect='auto')
        ax.set_xticks(range(len(pivot.columns)))
        ax.set_xticklabels(pivot.columns, rotation=45, ha='right')
        ax.set_yticks(range(len(pivot.index)))
        ax.set_yticklabels(pivot.index, fontsize=8)
        ax.set_title('Electrolyte Formulation vs Electrode (Cross-Bridge Scores)')
        plt.colorbar(im, label='Score')

        for i in range(len(pivot.index)):
            for j in range(len(pivot.columns)):
                val = pivot.values[i, j]
                if not np.isnan(val):
                    ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7)

        plt.tight_layout()
        plt.show()